# 04 — Model based grading

When building prompt evaluation workflows, grading systems provide objective signals about output quality. A grader takes model output and returns some kind of measurable feedback - typically a number between 1 and 10, where 10 represents high quality and 1 represents poor quality.

In [1]:
# Add repo root to path
import sys
sys.path.append("../..")

In [2]:
# Get the client and model from utils
from src.utils import get_client, get_model

client = get_client()
model = get_model()

In [3]:
# Helper functions
from src.utils import chat, add_user_message, add_assistant_message

There are three main approaches to grading model outputs:

Code graders - Programmatically evaluate outputs using custom logic
Model graders - Use another AI model to assess the quality
Human graders - Have people manually review and score outputs

### Code Graders
Code graders let you implement any programmatic check you can imagine. Common uses include:

- Checking output length
- Verifying output does/doesn't have certain words
- Syntax validation for JSON, Python, or regex
- Readability scores

The only requirement is that your code returns some usable signal - usually a number between 1 and 10.

### Model Graders
Model graders feed your original output into another API call for evaluation. This approach offers tremendous flexibility for assessing:

- Response quality
- Quality of instruction following
- Completeness
- Helpfulness
- Safety

### Human Graders
Human graders provide the most flexibility but are time-consuming and tedious. They're useful for evaluating:

- General response quality
- Comprehensiveness
- Depth
- Conciseness
- Relevance

## 1. The model grader

In [4]:
import json

In [5]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(
        messages=messages,
        client=client,
        model=model,
        stop_sequences=["```"]
    )
    return json.loads(eval_text)

The key insight is asking for strengths, weaknesses, and reasoning alongside the score. Without this context, models tend to default to middling scores around 6.

## 2. Integrating Grading into Your Workflow

In [6]:
def run_prompt(test_case: dict[str, str]) -> str:
    """
    Merges the prompt and test case input, then returns the result
    
    Args:
        test_case (dict[str, str]): A dictionary containing the task and any
            additional information needed to solve the task.
    
    Returns:
        str: The output generated by the model after running the prompt.
    """
    
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages=messages, content=prompt)
    output = chat(
        messages=messages,
        client=client,
        model=model,
        temperature=0.0,
    )
    return output

In [7]:
def run_test_case(test_case: dict[str, str]) -> dict[str, str | int | dict]:
    """
    Calls run_prompt, then grades the result
    
    Args:
        test_case (dict[str, str]): A dictionary containing the task and any
            additional information needed to solve the task.
    
    Returns:
        dict[str, str | int | dict]: A dictionary containing the output, test case, and score.
    """
    output = run_prompt(test_case)
    
    # Grade the output using the model
    evaluation = grade_by_model(test_case, output)
    score = evaluation["score"]
    reasoning = evaluation["reasoning"]
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [8]:
from statistics import mean

In [9]:
def run_eval(dataset: list[dict[str, str]]) -> list[dict[str, str | int | dict]]:
    """
    Loads the dataset and calls run_test_case with each case
    
    Args:
        dataset (list[dict[str, str]]): A list of dictionaries, each containing a task and any additional information needed to solve the task.
    
    Returns:
        list[dict[str, str | int | dict]]: A list of dictionaries, each containing the output, test case, and score.
    """
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean(result["score"] for result in results)
    print(f"Average Score: {average_score}")
    
    return results

In [10]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score: 6


In [11]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Here's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\ndef extract_region_from_s3_url(s3_url):\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Args:\n        s3_url (str): S3 URL in the format 's3://bucket-name.region.amazonaws.com'\n    \n    Returns:\n        str: The AWS region name, or None if the URL format is invalid\n    \n    Examples:\n        >>> extract_region_from_s3_url('s3://my-bucket.us-west-2.amazonaws.com')\n        'us-west-2'\n        >>> extract_region_from_s3_url('s3://example-bucket.eu-central-1.amazonaws.com')\n        'eu-central-1'\n    \"\"\"\n    try:\n        # Remove the 's3://' prefix\n        if not s3_url.startswith('s3://'):\n            return None\n        \n        # Extract the hostname part\n        hostname = s3_url[5:]  # Remove 's3://'\n        \n        # Split by dots to get parts\n        parts = hostname.split('.')\n        \n        # Expected format: bucket

This gives you an objective metric to track as you iterate on your prompt. While model graders can be somewhat capricious, they provide a consistent baseline for measuring improvements.